In [ ]:
# from LadrunoGraphStyle import set_default_plot_params, main_colors
# set_default_plot_params()

In [ ]:
from fem import (
    # Core
    Material,
    # Sections
    Membrane,
    # Gmsh tools
    GMSHtools,
    # Model
    FEMModel,
    # Elements
    CST, LST, Quad4, Quad9,
    # Units
    kgf, tf, kN, MPa, GPa, kg, g, cm, m, mm,
    # Parameters
    globalParameters,
)


import os
import numpy as np
import matplotlib.pyplot as plt
import gmsh
np.set_printoptions(suppress=True, precision=6, linewidth=400)

In [ ]:
globalParameters['nDoF'] = 2
globalParameters['nDIM'] = 2

In [ ]:
# General model parameters
L = 5000 
H = 500  
B = 300
lc = 100
output_path = os.getcwd()
mesh_name = 'calibrated_beam'
output_file = os.path.join(output_path, mesh_name + '.msh')

if not os.path.exists(output_path):
    os.makedirs(output_path)

In [ ]:
# Create a new GMSH model
gmsh.initialize()
gmsh.model.add(mesh_name)

# Corner points
p1 = gmsh.model.geo.addPoint(0,   0, 0, lc)
p2 = gmsh.model.geo.addPoint(L,   0, 0, lc)
p3 = gmsh.model.geo.addPoint(L,   H, 0, lc)
p4 = gmsh.model.geo.addPoint(0,   H, 0, lc)

# Mid points (bottom and top)
p5 = gmsh.model.geo.addPoint(L/2, 0, 0, lc)
p6 = gmsh.model.geo.addPoint(L/2, H, 0, lc)

# Lines
l1 = gmsh.model.geo.addLine(p1, p5)  # Bottom left
l2 = gmsh.model.geo.addLine(p5, p2)  # Bottom right
l3 = gmsh.model.geo.addLine(p2, p3)  # Right side
l4 = gmsh.model.geo.addLine(p3, p6)  # Top right
l5 = gmsh.model.geo.addLine(p6, p4)  # Top left
l6 = gmsh.model.geo.addLine(p4, p1)  # Left side
l7 = gmsh.model.geo.addLine(p5, p6)  # Center vertical

# Surfaces
c1 = gmsh.model.geo.addCurveLoop([l1,  l7, l5, l6])  # Left half
s1 = gmsh.model.geo.addPlaneSurface([c1])

c2 = gmsh.model.geo.addCurveLoop([l2, l3, l4, -l7])  # Right half
s2 = gmsh.model.geo.addPlaneSurface([c2])

gmsh.model.geo.synchronize()

# Physical groups
gmsh.model.addPhysicalGroup(2, [s1, s2], 201)
gmsh.model.setPhysicalName(2, 201, "Beam")

gmsh.model.addPhysicalGroup(0, [p1], 101)
gmsh.model.setPhysicalName(0, 101, "Support_left")

gmsh.model.addPhysicalGroup(0, [p2], 102)
gmsh.model.setPhysicalName(0, 102, "Support_right")

gmsh.model.addPhysicalGroup(0, [p6], 50)
gmsh.model.setPhysicalName(0, 50, "Load")

gmsh.model.geo.synchronize()

def all_quads():
    gmsh.model.geo.synchronize()
    gmsh.option.setNumber("Mesh.RecombineAll", 1)
    gmsh.option.setNumber("Mesh.Algorithm", 8)
    gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 2)
    gmsh.option.setNumber("Mesh.Smoothing", 100)
    gmsh.option.setNumber("Mesh.ElementOrder", 1)

all_quads()
# gmsh.option.setNumber("Mesh.ElementOrder", 2)


In [98]:
# Generate the mesh
gmsh.model.mesh.generate()
# Save mesh to disk
gmsh.write(output_file)
# Open GMSH GUI
gmsh.fltk.run()
# Write mesh file again (redundant)
v = gmsh.write(output_file)
# Close GMSH instance
gmsh.finalize()

In [ ]:
# Define material
fc=210
fc_21 = Material(name='fc_21',
                 E=13500*fc**0.5*kgf/cm**2,
                 nu=0.20,
                 rho= 2400*kg/m**3*g*0)

# Define membrane section
ConcreteBeam = Membrane(name='ConcreteBeam',
                        thickness=300,
                        material=fc_21)

# Map physical group IDs to sections
section_dictionary = {201: ConcreteBeam}

# Define distributed load magnitude
Pload = 10*tf

load_dictionary = {
    50:  {'value': Pload, 'direction': '-y'},   
}

# Define boundary conditions (restraints)
restrain_dictionary = {101: ['r', 'r'],
                       102: ['f', 'r']}


In [ ]:
# read mesh 
mesh = GMSHtools(output_file)

In [ ]:
mesh.node_map

In [ ]:
# por tag de gmsh
node = mesh.node_map[6]
print(node)

# o ver sus propiedades
node.print_summary()

In [ ]:
mesh.elements

In [ ]:
mesh.physical_groups["Beam"].elements


In [ ]:
# Map number of nodes per element to element class
element_map = {
    3: CST,         # 3-node triangles
    4: Quad4,       # 4-node quadrilaterals
    6: LST,         # 6-node triangles
    9: Quad9,       # 9-node quadrilaterals
}

# Build FEM model — applies BCs, builds elements, assembles load vector
model = FEMModel(
    mesh                = mesh,
    section_dictionary  = section_dictionary,
    restrain_dictionary = restrain_dictionary,
    load_dictionary     = load_dictionary,
    element_class_map   = element_map,
    analysis_type       = 'planeStress',
    consistent_loads    = False,
    sampling_points     = 3,

    verbose=True,
)

In [ ]:
model.F_nodal

In [ ]:
model.F_load

In [ ]:
model.system_nDof

In [ ]:
# Mesh diagnostics
model.check_mesh()

In [ ]:
# Solve — single step
model.solve_static(
    n_steps=1, 
    load_factor=1.0,
    verbose = True,
)

In [ ]:
# Get node by tag and location
model.get_node(tags=[6, 10], 
               locations=[(2500, 250)],
               step = -1 , 
               );


In [ ]:
# Get element by tag and location
model.get_element(tags=[6, 10], 
               locations=[(2500, 250)],
               step = -1 , 
               );


In [ ]:
# Node history — evolution over steps
uy_history = model.node_history(
    tag       = 6,
    component = 'uy',   # 'ux', 'uy', 'uz'
)

In [ ]:
# Element history
sxx_history = model.element_history(
    tag       = 10,
    component = 'sxx',  # 'sxx','syy','sxy','exx','eyy','exy','vm'
)

In [ ]:
# Plot mesh
model.plot(
    step                = -1,
    show_node_labels    = False,
    show_element_labels = False,
    show_supports       = True,
    show_element_edges  = True,
    show_node_points    = True,
    figsize             = (12, 8),
)

In [ ]:
# Plot loads
model.plot_loads(
    show_element_edges = True,
    show_node_points   = True,
    show_supports      = True,
    figsize            = (12, 8),
)

In [ ]:
# Plot deformed
model.plot_deformed(
    sfac                = 50,
    step                = -1,
    component           = 'umag',   # 'ux', 'uy', 'umag'
    cmap                = 'viridis',
    show_element_edges  = True,
    show_supports       = True,
    figsize             = (12, 8),
)

In [ ]:
# Plot field
model.plot_field(
    component           = 'sxx',        # 'sxx','syy','sxy','vmis','s1','s2','exx','eyy','exy'
    source              = 'fem',
    step                = -1,
    result_type         = 'nodal_avg',  # 'nodal_avg' or 'element'
    deformed            = True,
    sfac                = 50,
    cmap                = 'turbo',
    show_element_edges  = True,
    show_supports       = True,
    figsize             = (12, 8),
)

In [ ]:
# Plot field
model.plot_field(
    component           = 'exy',        # 'sxx','syy','sxy','vmis','s1','s2','exx','eyy','exy'
    source              = 'fem',
    step                = -1,
    result_type         = 'nodal_avg',  # 'nodal_avg' or 'element'
    deformed            = True,
    sfac                = 50,
    cmap                = 'turbo',
    show_element_edges  = True,
    show_supports       = True,
    figsize             = (12, 8),
)

In [ ]:
# Send to gmsh — static
model.plot2gmsh(
    step            = -1,
    source         = 'fem',
    disp_factor     = 50,
    show_disp       = True,
    show_loads      = True,
    show_reactions  = True,
    show_stress     = True,
    show_strain     = True,
    show_vm         = True,
    show_averaged   = False,

)

In [ ]:
# Send to gmsh — animation of all steps
model.plot2gmsh_animate(
    disp_factor=10
)

In [ ]:
# Save results to HDF5
output_file_h5 = os.path.join(output_path, mesh_name + '.h5')
model.save(output_file_h5)

In [ ]:
# Load results from HDF5
results = FEMModel.load_results(output_file_h5)

In [ ]:
# por tags
model.plot_node_history(tags=[6, 10, 15], 
                        locations=[(2500, 250), (0, 250)],
                        component='uy')


# elementos
model.plot_element_history(tags=[10, 20], 
                           locations=[(2500, 0), (1000, 250)],
                           component='sxx')


## Opensees

In [ ]:
import openseespy.opensees as ops

# import sys
# sys.path.insert(0, r"C:\Dropbox\01. Brain\11. GitHub\OpenSees\compile_windows\build-win11")

# import opensees as ops
# print(ops.__file__)



import opsvis as opsv

ops.wipe()
ops.model('basicBuilder','-ndm',2,'-ndf',2)

In [ ]:
# Nodes
for tag, (x, y, z) in mesh.nodes.items():
    ops.node(tag, x, y)

In [ ]:
# Boundary conditions
fixed_nodes = set()
for tag in mesh.physical_groups['Support_left'].nodes:
    if tag not in fixed_nodes:
        fixed_nodes.add(tag)
        # ops.fix(tag, 1, 1, 1, 1, 1, 1)
        ops.fix(tag, 1, 1)

In [ ]:
# Boundary conditions
fixed_nodes = set()
for tag in mesh.physical_groups['Support_right'].nodes:
    if tag not in fixed_nodes:
        fixed_nodes.add(tag)
        # ops.fix(tag, 1, 1, 1, 1, 1, 1)
        ops.fix(tag, 0, 1)

In [ ]:
# Material
solidMaterialTag = 1

ops.nDMaterial('ElasticIsotropic', solidMaterialTag,
               fc_21.E,
               fc_21.nu,
               fc_21.rho)

In [ ]:
group = mesh.physical_groups['Beam'].elements
for elem_tag, conn in zip(group['element_tags'], group['connectivity']):
    n1, n2, n3 = conn
    ops.element('tri31', elem_tag, n1, n2, n3, ConcreteBeam.thickness, 'PlaneStress', solidMaterialTag)

In [ ]:
opsv.plot_model(node_labels=0, element_labels=0, fig_wi_he=(50,25))

In [ ]:
# Loads
ts_tag      = 2
pattern_tag = 2
ops.timeSeries('Linear', ts_tag)
ops.pattern('Plain', pattern_tag, ts_tag)

for tag, force in model.F_nodal.items():
    if np.any(np.abs(force) > 0):
        ops.load(tag, *force.tolist())

In [ ]:
NstepGravity=10
DGravity=1/NstepGravity

ops.system("FullGeneral")
ops.numberer("Plain")
ops.constraints("Plain")
ops.integrator("LoadControl", DGravity )
ops.test("NormUnbalance", 1.0e-6, 100 , 0)
ops.algorithm("Newton")
ops.analysis("Static")


# ops.recorder('mpco', 'test.mpco', '-N', 'displacement')

ops.analyze(NstepGravity)

In [ ]:

# ops.remove('recorders')
# ops.wipe()


In [ ]:
opsv.plot_defo(
    sfac=10,
    fig_wi_he=(50, 25),
    endDispFlag=True,
    unDefoFlag=0
)

In [ ]:
jstr = 'sxx'
# jstr = 'syy'
# jstr = 'sxy'
# jstr = 'vmis'
# jstr = 's1'
# jstr = 's2'
# jstr = 'angle'

plt.figure(figsize=(15, 8))
opsv.plot_stress(jstr)
plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.title(f'{jstr}')

In [ ]:
# Extract OpenSees results into model
model.set_results_opensees(
    ops              = ops,
    step             = 0,
)

# Visualize in gmsh
model.plot2gmsh(
    step           = -1,
    source         = 'opensees',
    disp_factor    = 50,
    show_disp      = True,
    show_loads     = True,
    show_reactions = True,
    show_stress    = True,
    show_strain    = True,
    show_vm        = True,
    show_averaged  = True,
)


In [ ]:
# What's in the model
print(repr(model))
print(f"\nFEM steps     : {len(model.results_fem)}")
print(f"OpenSees steps: {len(model.results_opensees)}")
print(f"Modal modes   : {len(model.results_opensees_modal) if hasattr(model, 'results_opensees_modal') else 0}")

# Modal summary
if hasattr(model, 'results_opensees_modal'):
    print(f"\n  {'Mode':>6}  {'Freq [Hz]':>12}  {'Period [s]':>12}")
    for mr in model.results_opensees_modal:
        print(f"  {mr.mode:>6}  {mr.freq:>12.4f}  {mr.period:>12.4f}")